# Qwen 2.5-7B Fabric Expert - Interactive Demo

Fine-tuned Qwen model for Hyperledger Fabric Q&A.
This interactive notebook allows you to test a Small Language Model (SLM) specifically optimized for Hyperledger Fabric.

**Model:** Qwen 2.5-7B-Instruct + LoRA adapters  
**GPU Required:** T4 (15GB VRAM) - **Free in Google Colab!**

**Training**: Fine-tuned using LoRA on a custom dataset of technical instructions regarding Hyperledger Fabric (Architecture, Chaincode, Lifecycle, Policies).

**Optimization**: The model is loaded in 4-bit precision to ensure speed and efficiency on standard hardware.

## Quick Instructions
Enable GPU: Go to Runtime > Change runtime type and ensure **T4 GPU** is selected.

**Execute Cells**: Click the "Play" button on each cell in sequential order.

**Ask a Question**: Use the final cell to input your query regarding Hyperledger Fabric.

**Note**: Initial setup may take 5-6 minutes to download the model weights from Hugging Face.

In [1]:
# 1. Install dependencies
!pip install -q transformers peft gradio torch accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 10.5 MB/s eta 0:00:00


In [ ]:
# 2. Load model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# clear cache
import gc
torch.cuda.empty_cache()
gc.collect()

model_id = "gcapuzzi/qwen2.5-7b-fabric-expert"

# Quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("Loading model in 4-bit (optimized fro T4)...")

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization_config,
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

model.eval()
print("Model loaded")

Loading model in 4-bit (optimized fro T4)...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/15.2G [00:00<?, ?B/s]

In [ ]:
# 3. Inference function
def ask_fabric_expert(question, max_tokens=1000, temperature=0.7):
    """Generate answer to Hyperledger Fabric question"""

    prompt = f"""<|im_start|>system
You are a Hyperledger Fabric expert.<|im_end|>
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda") # use GPU

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("assistant")[-1].strip()

In [ ]:
# 4. Test
print(ask_fabric_expert("What is a chaincode?"))

In [ ]:
# 5. Launch Gradio UI
import gradio as gr

demo = gr.Interface(
    fn=ask_fabric_expert,
    inputs=[
        gr.Textbox(label="Question", placeholder="What is a chaincode?", lines=2),
        gr.Slider(100, 1000, value=300, label="Max Tokens"),
        gr.Slider(0.1, 1.5, value=0.7, label="Temperature")
    ],
    outputs=gr.Textbox(label="Answer", lines=10),
    title="Hyperledger Fabric Expert AI",
    description="Ask questions about Hyperledger Fabric. Powered by fine-tuned Qwen 2.5-7B.",
    examples=[
        ["What is a chaincode?", 1000, 0.7],
        ["How do I create a channel in Fabric v2.5?", 500, 0.5],
        ["Explain the difference between BFT and CFT ordering", 400, 0.7],
        ["I'm getting MVCC_READ_CONFLICT errors. How do I fix this?", 400, 0.6]
    ],
    cache_examples=False
)

demo.launch(share=True)  # Creates public URL